# Session 32: Logistic Regression Classification

This notebook implements Logistic Regression for at-risk classification.
- 1 = at-risk (G3 < 10)
- 0 = successful (G3 >= 10)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Load data

In [ ]:
data_dir = Path("data/processed")
X_full = pd.read_parquet(data_dir / "X_full.parquet")
y = pd.read_parquet(data_dir / "y_full.parquet")
y = y["G3"] if "G3" in y.columns else y.iloc[:, 0]
print(f"Features: {X_full.shape}, Target: {len(y)}")

## 3. Train/test split and classification labels

In [ ]:
Xtr_f, Xte_f, ytr, yte = train_test_split(X_full, y, test_size=0.20, random_state=42)
yc = (y < 10).astype(int)
yctr = yc.loc[ytr.index]
ycte = yc.loc[yte.index]
print("Training:", yctr.value_counts().to_dict())
print("Test:", ycte.value_counts().to_dict())

## 4. Fit Logistic Regression

In [ ]:
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
clf.fit(Xtr_f, yctr)
y_pred = clf.predict(Xte_f)
y_proba = clf.predict_proba(Xte_f)[:, 1]

print("Accuracy:", accuracy_score(ycte, y_pred))
print("Precision:", precision_score(ycte, y_pred))
print("Recall:", recall_score(ycte, y_pred))
print("F1:", f1_score(ycte, y_pred))
print("ROC_AUC:", roc_auc_score(ycte, y_proba))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(ycte, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Successful", "At-risk"])
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.show()

## 6. Classification Report

In [ ]:
print(classification_report(ycte, y_pred, target_names=["Successful", "At-risk"]))

## 7. Coefficients

In [ ]:
coef_df = pd.DataFrame({
    "Feature": Xtr_f.columns,
    "Coefficient": clf.named_steps["logisticregression"].coef_[0]
})
coef_df["Abs"] = coef_df["Coefficient"].abs()
display(coef_df.sort_values("Abs", ascending=False).head(10))

## 8. Classification Row

In [ ]:
tn, fp, fn, tp = cm.ravel()
classification_row = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(ycte, y_pred),
    "Precision": precision_score(ycte, y_pred),
    "Recall": recall_score(ycte, y_pred),
    "F1": f1_score(ycte, y_pred),
    "ROC_AUC": roc_auc_score(ycte, y_proba),
    "True_Negative": tn,
    "False_Positive": fp,
    "False_Negative": fn,
    "True_Positive": tp
}])
display(classification_row)

## Session 32 Complete

✅ Logistic Regression with StandardScaler
✅ Classification metrics computed
✅ Confusion matrix displayed
✅ Classification row created

## Session 33: KNN, SVM, and Naive Bayes Classification

This section adds three classifiers: K-Nearest Neighbors, Support Vector Machine, and Gaussian Naive Bayes.
All classifiers use the same training and test data with scaling for KNN and SVM.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Verify prerequisites
for obj in ["Xtr_f", "Xte_f", "yctr", "ycte"]:
    if obj not in globals():
        raise NameError(f"Missing required object: {obj}")

print("Session 33 prerequisites verified")
print("Training shape:", Xtr_f.shape)
print("Test shape:", Xte_f.shape)


In [ ]:
def evaluate_classifier(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
    }
print("Evaluator ready")


In [ ]:
# Train KNN, SVM, and Gaussian Naive Bayes
results = []
for code, name, family, clf in [
    ("KNN", "K-Nearest Neighbors", "Instance-based", KNeighborsClassifier()),
    ("SVM", "Support Vector Machine", "Maximum-margin", SVC(probability=True, random_state=42)),
    ("NB", "Gaussian Naive Bayes", "Probabilistic", GaussianNB()),
]:
    pipeline = make_pipeline(StandardScaler(), clf)
    pipeline.fit(Xtr_f, yctr)
    predictions = pipeline.predict(Xte_f)
    probabilities = pipeline.predict_proba(Xte_f)[:, 1]
    metrics = evaluate_classifier(ycte, predictions, probabilities)
    results.append({
        "Model": code,
        "Full_Model_Name": name,
        "Model_Family": family,
        "Scaling_Used": True,
        **metrics
    })
    print(f"{code} completed - F1: {metrics["f1"]:.4f}")

results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
results_df.insert(0, "Session33_F1_Rank", range(1, len(results_df) + 1))

print("\nSession 33 Classification Results:")
print(results_df.to_string())


In [ ]:
# Save results to CSV
from pathlib import Path
repo_root = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists()), Path.cwd())
output_dir = repo_root / "reports" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(output_dir / "session33_classification_rows.csv", index=False)
print("Results saved to:", output_dir / "session33_classification_rows.csv")


### Session 33 Interpretation

- **KNN** classifies based on similarity in scaled feature space
- **SVM** finds a maximum-margin decision boundary
- **Gaussian Naive Bayes** assumes conditional independence of predictors

Naive Bayes provides a useful baseline even with the independence assumption.


# Session 36: Gradient Boosting and AdaBoost
**GitHub deliverable:** Add Gradient Boosting and AdaBoost classifiers to the classification notebook.

In [ ]:
# SESSION_36_BOOSTING_GITHUB_DELIVERABLE
import time
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

required_session36_variables = ["Xtr_f", "Xte_f", "yctr", "ycte"]
missing_session36_variables = [v for v in required_session36_variables if v not in globals()]
if missing_session36_variables:
    print("Note: Run earlier classification cells first to define:", missing_session36_variables)


In [ ]:
boosting_models_s36 = {"GradBoost": GradientBoostingClassifier(random_state=42), "AdaBoost": AdaBoostClassifier(random_state=42)}
print("Session 36 Boosting Models:", list(boosting_models_s36.keys()))


In [ ]:
if "classification_leaderboard" not in globals():
    classification_leaderboard = pd.DataFrame()
print("Leaderboard initialized for Session 36.")
